# Baseline Comparison — Does the VAE Earn Its Complexity?

Input: `data/processed/windows_cc1/X_*.npy` (same whitened PCA features the VAE
uses — a fair comparison isolates the *model*, not the *features*).

**Why this notebook exists:** every result so far validates the VAE+PCA approach
against itself (ablations, thresholding variants) — never against something
simpler. This is the most likely first question an examiner asks: *"how do you
know this needed a VAE?"*

**Two baselines, same leak-free protocol as `vae_eval.ipynb`** (threshold chosen
only from `cc1_val`'s unsupervised false-positive rate, never touching test/drift
labels):

1. **Gaussian / whitened-distance baseline** — no training at all beyond the
   already-fit PCA whitening. Since PCA output is whitened (mean~0, std~1 per
   component, verified in `windowing_pca.ipynb`), a `cc1_train`-normal point
   should look like a draw from `N(0, I)`. Anomaly score = `||x||^2` (sum of
   squared components) — literally distance from the origin in whitened space.
   The simplest possible statistical baseline with zero learned parameters.
2. **Isolation Forest** — a standard, no-deep-learning classical anomaly
   detector (`sklearn.ensemble.IsolationForest`), fit on `cc1_train` only, same
   whitened PCA features.

In [1]:
import numpy as np
import pickle, os
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, confusion_matrix,
)

BASE     = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
WIN_DIR  = os.path.join(BASE, 'data', 'processed', 'windows_cc1')
MODEL_DIR = os.path.join(BASE, 'models')

SETS = ['cc1_train', 'cc1_val', 'cc1_test', 'drift_sc1', 'drift_sc2', 'drift_cc2']
DRIFT_SETS = ['drift_sc1', 'drift_sc2', 'drift_cc2']

X, y = {}, {}
for name in SETS:
    X[name] = np.load(os.path.join(WIN_DIR, f'X_{name}.npy'))
    y[name] = np.load(os.path.join(WIN_DIR, f'y_{name}.npy'))
    print(f'  {name:10s}: {X[name].shape}  anomalies={int(y[name].sum()):,}')

vae_eval = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_eval.pkl'), 'rb'))
print('\nVAE reference results loaded for comparison.')

  cc1_train : (154198, 26)  anomalies=0
  cc1_val   : (21573, 26)  anomalies=0
  cc1_test  : (44185, 26)  anomalies=256
  drift_sc1 : (59697, 26)  anomalies=134
  drift_sc2 : (38070, 26)  anomalies=85
  drift_cc2 : (76977, 26)  anomalies=720

VAE reference results loaded for comparison.


## Baseline A — Gaussian / whitened-distance score

`score(x) = sum(x_i^2)`. No fitting needed — the PCA whitening already IS the
model. This tests whether the VAE's nonlinear reconstruction is doing anything
that simple per-component squared distance from the training mean couldn't.

In [2]:
def gaussian_score(Xarr):
    return (Xarr ** 2).sum(axis=1)

scores_gaussian = {name: gaussian_score(X[name]) for name in SETS}

print('cc1_train whitening sanity check (should be close to 0 mean / 1 std per component):')
print(f'  mean of per-component mean: {X["cc1_train"].mean():.4f}')
print(f'  mean of per-component std:  {X["cc1_train"].std():.4f}')

cc1_train whitening sanity check (should be close to 0 mean / 1 std per component):
  mean of per-component mean: 0.0000
  mean of per-component std:  1.0000


## Baseline B — Isolation Forest

Fit on `cc1_train` only (same training set the VAE uses), same whitened PCA
features. `n_estimators=100` (sklearn default-adjacent), `contamination='auto'`
(doesn't use labels — purely structural).

In [3]:
iso_forest = IsolationForest(n_estimators=100, contamination='auto', random_state=42, n_jobs=-1)
iso_forest.fit(X['cc1_train'])

# decision_function: higher = more normal. Negate so higher = more anomalous, matching every other score in this project.
scores_iso = {name: -iso_forest.decision_function(X[name]) for name in SETS}
print('Isolation Forest fit on cc1_train.')

Isolation Forest fit on cc1_train.


## Leak-free thresholds for each baseline (from `cc1_val` only, same discipline as `vae_eval.ipynb`)

In [4]:
thresh_gaussian = float(np.percentile(scores_gaussian['cc1_val'], 99))
thresh_iso      = float(np.percentile(scores_iso['cc1_val'], 99))
print(f'Gaussian baseline val_p99 threshold: {thresh_gaussian:.4f}')
print(f'Isolation Forest val_p99 threshold:  {thresh_iso:.4f}')

Gaussian baseline val_p99 threshold: 85.5405
Isolation Forest val_p99 threshold:  0.0375


## Full comparison: VAE vs. Gaussian vs. Isolation Forest, on every set

In [5]:
def evaluate(scores, y_true, threshold):
    auc_roc = roc_auc_score(y_true, scores)
    auc_pr  = average_precision_score(y_true, scores)
    pred = (scores > threshold).astype(int)
    p = precision_score(y_true, pred, zero_division=0)
    r = recall_score(y_true, pred, zero_division=0)
    f1 = f1_score(y_true, pred, zero_division=0)
    return {'auc_roc': auc_roc, 'auc_pr': auc_pr, 'precision': p, 'recall': r, 'f1': f1}

results = {'gaussian': {}, 'isolation_forest': {}, 'vae': {}}
for name in ['cc1_test'] + DRIFT_SETS:
    results['gaussian'][name] = evaluate(scores_gaussian[name], y[name], thresh_gaussian)
    results['isolation_forest'][name] = evaluate(scores_iso[name], y[name], thresh_iso)
    results['vae'][name] = {
        'auc_roc': vae_eval['auc'][name]['auc_roc'],
        'auc_pr':  vae_eval['auc'][name]['auc_pr'],
        'precision': vae_eval['precision_recall'][name]['val_p99']['precision'],
        'recall':    vae_eval['precision_recall'][name]['val_p99']['recall'],
        'f1':        vae_eval['precision_recall'][name]['val_p99']['f1'],
    }

print(f'{"set":12s} {"method":18s} {"AUC-ROC":>9s} {"AUC-PR":>8s} {"precision":>10s} {"recall":>8s} {"F1":>7s}')
for name in ['cc1_test'] + DRIFT_SETS:
    for method in ['gaussian', 'isolation_forest', 'vae']:
        r = results[method][name]
        print(f'{name:12s} {method:18s} {r["auc_roc"]:9.4f} {r["auc_pr"]:8.4f} {r["precision"]:10.3f} {r["recall"]:8.3f} {r["f1"]:7.3f}')
    print()

set          method               AUC-ROC   AUC-PR  precision   recall      F1
cc1_test     gaussian              0.8798   0.6554      0.694    0.637   0.664
cc1_test     isolation_forest      0.7584   0.1135      0.138    0.227   0.171
cc1_test     vae                   0.8763   0.6014      0.630    0.605   0.618

drift_sc1    gaussian              0.8962   0.0556      0.013    0.858   0.025
drift_sc1    isolation_forest      0.8113   0.0529      0.015    0.567   0.030
drift_sc1    vae                   0.8515   0.0181      0.020    0.806   0.040

drift_sc2    gaussian              0.5654   0.3889      0.006    0.541   0.012
drift_sc2    isolation_forest      0.2156   0.0013      0.000    0.000   0.000
drift_sc2    vae                   0.6327   0.0615      0.012    0.482   0.024

drift_cc2    gaussian              0.8755   0.1430      0.090    0.739   0.161
drift_cc2    isolation_forest      0.8348   0.2507      0.146    0.443   0.220
drift_cc2    vae                   0.8812   0.408

## Save results

In [6]:
out_path = os.path.join(MODEL_DIR, 'baseline_comparison.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(results, f)
print(f'Saved -> {out_path}')

Saved -> c:\Users\jthar\Documents\Claude\Projects\module3\models\baseline_comparison.pkl


## How to read this

If the VAE's F1/AUC-PR clearly beats both baselines on `cc1_test` (in-distribution),
that justifies the added complexity for the core task. If a baseline is
competitive or better on a specific *drift* set, that's worth reporting honestly —
it would mean the VAE's nonlinear modeling isn't adding value there specifically,
which is itself a legitimate, useful finding (not a failure of this notebook).